In [14]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import src.preprocess
importlib.reload(src.preprocess)
from src.preprocess import load_raw, engineer_features, encode_and_split

raw        = load_raw(PROJECT_ROOT / 'data/DataCoSupplyChainDataset.csv')
engineered = engineer_features(raw)

X_train, X_test, y_train, y_test, feat_names, encoders = encode_and_split(
    engineered, target_col='Late_delivery_risk', test_year=2017
)

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test positive rate:  {y_test.mean():.3f}')

Train: (125200, 37)
Test:  (55319, 37)
Train positive rate: 0.550
Test positive rate:  0.545


In [15]:
import mlflow
import mlflow.xgboost
from src.model import train_model, evaluate_model, save_model
import importlib, src.model
importlib.reload(src.model)
from src.model import train_model, evaluate_model, save_model

mlflow.xgboost.autolog()

with mlflow.start_run(run_name='xgb_supplyguard_v1'):
    model  = train_model(X_train, y_train, feat_names, tune=True)
    y_prob = evaluate_model(model, X_test, y_test)
    save_model(model, str(PROJECT_ROOT / 'models/xgb_model.pkl'))

Fitting 3 folds for each of 20 candidates, totalling 60 fits


2026/06/13 21:11:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'subsample': 0.9, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
=== Model Evaluation ===
AUC-ROC:  0.7636
Accuracy: 0.7019

              precision    recall  f1-score   support

     On-Time       0.63      0.86      0.72     25160
        Late       0.83      0.57      0.68     30159

    accuracy                           0.70     55319
   macro avg       0.73      0.71      0.70     55319
weighted avg       0.74      0.70      0.70     55319

Confusion Matrix:
[[21605  3555]
 [12938 17221]]
Model saved to D:\Supply Guard AI\models\xgb_model.pkl


In [16]:
import json, joblib

with open(PROJECT_ROOT / 'models/feature_names.json', 'w') as f:
    json.dump(feat_names, f)

joblib.dump(encoders, PROJECT_ROOT / 'models/encoders.pkl')

print('Saved:')
print(f'  models/xgb_model.pkl')
print(f'  models/feature_names.json')
print(f'  models/encoders.pkl')

Saved:
  models/xgb_model.pkl
  models/feature_names.json
  models/encoders.pkl


In [17]:
# DIAGNOSTIC CELL — run this standalone, no retraining needed
import pandas as pd
import numpy as np
import joblib, json
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import load_raw, engineer_features, encode_and_split

# Load saved model
model      = joblib.load(PROJECT_ROOT / 'models/xgb_model.pkl')
feat_names = json.load(open(PROJECT_ROOT / 'models/feature_names.json'))

# Rebuild test set
raw        = load_raw(PROJECT_ROOT / 'data/DataCoSupplyChainDataset.csv')
engineered = engineer_features(raw)
X_train, X_test, y_train, y_test, _, _ = encode_and_split(
    engineered, target_col='Late_delivery_risk', test_year=2017
)

# Check 1 — features
print('=== FEATURES IN MODEL ===')
for i, name in enumerate(feat_names):
    print(f'  {i}: {name}')
print(f'\nTotal: {len(feat_names)}')

# Check 2 — split sizes
print('\n=== SPLIT INFO ===')
print(f'Train size: {len(y_train):,}  |  positive rate: {y_train.mean():.3f}')
print(f'Test size:  {len(y_test):,}  |  positive rate:  {y_test.mean():.3f}')

# Check 3 — feature importances
print('\n=== TOP 15 FEATURE IMPORTANCES ===')
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
print(feat_imp.head(15))

# Check 4 — is scheduled days in features?
print('\n=== KEY COLUMN CHECK ===')
print('Days for shipment (scheduled) in features:', 'Days for shipment (scheduled)' in feat_names)

=== FEATURES IN MODEL ===
  0: Type
  1: Days for shipment (scheduled)
  2: Benefit per order
  3: Sales per customer
  4: Category Id
  5: Category Name
  6: Customer Country
  7: Customer Segment
  8: Customer State
  9: Department Id
  10: Department Name
  11: Latitude
  12: Longitude
  13: Market
  14: Order Country
  15: Order Item Cardprod Id
  16: Order Item Discount
  17: Order Item Discount Rate
  18: Order Item Product Price
  19: Order Item Profit Ratio
  20: Order Item Quantity
  21: Sales
  22: Order Item Total
  23: Order Profit Per Order
  24: Order Region
  25: Order Status
  26: Order Zipcode
  27: Product Card Id
  28: Product Category Id
  29: Product Price
  30: Product Status
  31: Shipping Mode
  32: order_month
  33: order_year
  34: order_quarter
  35: month_sin
  36: month_cos

Total: 37

=== SPLIT INFO ===
Train size: 125,200  |  positive rate: 0.550
Test size:  55,319  |  positive rate:  0.545

=== TOP 15 FEATURE IMPORTANCES ===
Days for shipment (scheduled)

In [18]:
print('=== TOP 15 FEATURE IMPORTANCES ===')
print(feat_imp.head(15).to_string())

print('\n=== BOTTOM 5 (least useful features) ===')
print(feat_imp.tail(5).to_string())

print('\n=== SCHEDULED DAYS IMPORTANCE ===')
print(f"Days for shipment (scheduled): {feat_imp.get('Days for shipment (scheduled)', 'NOT FOUND')}")
print(f"Shipping Mode importance:      {feat_imp.get('Shipping Mode', 'NOT FOUND')}")
print(f"Order Region importance:       {feat_imp.get('Order Region', 'NOT FOUND')}")

=== TOP 15 FEATURE IMPORTANCES ===
Days for shipment (scheduled)    0.443412
Shipping Mode                    0.288583
Order Status                     0.081445
Type                             0.013163
Market                           0.008803
Customer Segment                 0.008436
month_cos                        0.008393
Order Country                    0.008092
month_sin                        0.007916
Order Region                     0.007784
Latitude                         0.007708
order_month                      0.007640
Order Zipcode                    0.007639
order_quarter                    0.007525
order_year                       0.007405

=== BOTTOM 5 (least useful features) ===
Product Price          0.003247
Order Item Total       0.003185
Department Id          0.000000
Product Category Id    0.000000
Product Status         0.000000

=== SCHEDULED DAYS IMPORTANCE ===
Days for shipment (scheduled): 0.44341230392456055
Shipping Mode importance:      0.28858330845832

In [19]:
print(raw['Order Status'].value_counts())

Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64


In [20]:
import pandas as pd
import numpy as np
import joblib, json
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

model      = joblib.load(PROJECT_ROOT / 'models/xgb_model.pkl')
feat_names = json.load(open(PROJECT_ROOT / 'models/feature_names.json'))

from src.preprocess import load_raw, engineer_features, encode_and_split
raw = load_raw(PROJECT_ROOT / 'data/DataCoSupplyChainDataset.csv')
engineered = engineer_features(raw)

X_train, X_test, y_train, y_test, _, _ = encode_and_split(
    engineered, target_col='Late_delivery_risk', test_year=2017
)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# Filter raw to test years using original date column
raw['order_year_temp'] = pd.to_datetime(
    raw['order date (DateOrders)'], errors='coerce'
).dt.year
test_raw = raw[raw['order_year_temp'] >= 2017].copy().reset_index(drop=True)
test_raw = test_raw.iloc[:len(y_test)].copy()

# Add predictions
test_raw['predicted_risk']  = y_prob
test_raw['predicted_label'] = y_pred
test_raw['actual_label']    = y_test
test_raw['error']           = y_test - y_pred

# false negative = model said on-time, actually late
# false positive = model said late, actually on-time
false_negatives = test_raw[test_raw['error'] == 1]
false_positives = test_raw[test_raw['error'] == -1]

print(f'Total test orders:    {len(test_raw):,}')
print(f'False negatives:      {len(false_negatives):,}  (missed late deliveries)')
print(f'False positives:      {len(false_positives):,}  (wrong late flags)')
print()

print('=== WHERE IS THE MODEL MISSING LATE DELIVERIES? ===')
print('\nBy Region:')
print(false_negatives['Order Region'].value_counts().head(8))

print('\nBy Shipping Mode:')
print(false_negatives['Shipping Mode'].value_counts())

print('\nBy Department:')
print(false_negatives['Department Name'].value_counts().head(8))

print('\nBy Month:')
print(false_negatives['order_month'].value_counts().sort_index()
      if 'order_month' in false_negatives.columns
      else pd.to_datetime(raw['order date (DateOrders)'], errors='coerce').dt.month
      [false_negatives.index].value_counts().sort_index())

Total test orders:    55,319
False negatives:      12,938  (missed late deliveries)
False positives:      3,555  (wrong late flags)

=== WHERE IS THE MODEL MISSING LATE DELIVERIES? ===

By Region:
Order Region
Central America    3622
Western Europe     2751
South America      1709
Northern Europe    1082
Southern Europe    1008
Caribbean           837
Southeast Asia      410
Oceania             319
Name: count, dtype: int64

By Shipping Mode:
Shipping Mode
Standard Class    11880
Same Day           1058
Name: count, dtype: int64

By Department:
Department Name
Fan Shop      4529
Apparel       3501
Golf          2041
Footwear       915
Outdoors       716
Discs Shop     476
Technology     253
Fitness        203
Name: count, dtype: int64

By Month:
order date (DateOrders)
1     1371
2      897
3      982
4      968
5     1139
6     1121
7     1113
8     1331
9     1082
10     945
11     960
12    1029
Name: count, dtype: int64


In [21]:
print('=== HIGHEST RISK COMBINATION (missed late deliveries) ===')
combo = false_negatives.groupby(
    ['Order Region', 'Shipping Mode']
).size().sort_values(ascending=False).head(10)
print(combo)

print('\n=== AS % OF TOTAL ORDERS IN THAT COMBINATION ===')
total_combo = test_raw.groupby(
    ['Order Region', 'Shipping Mode']
).size()
miss_rate = (combo / total_combo).sort_values(ascending=False).head(10)
print(miss_rate.round(3))

=== HIGHEST RISK COMBINATION (missed late deliveries) ===
Order Region     Shipping Mode 
Central America  Standard Class    3397
Western Europe   Standard Class    2489
South America    Standard Class    1590
Northern Europe  Standard Class     972
Southern Europe  Standard Class     872
Caribbean        Standard Class     792
Southeast Asia   Standard Class     379
Oceania          Standard Class     287
Eastern Asia     Standard Class     286
Western Europe   Same Day           262
dtype: int64

=== AS % OF TOTAL ORDERS IN THAT COMBINATION ===
Order Region     Shipping Mode 
Central America  Standard Class    0.384
Western Europe   Same Day          0.379
South America    Standard Class    0.367
Eastern Asia     Standard Class    0.362
Northern Europe  Standard Class    0.358
Southeast Asia   Standard Class    0.358
Southern Europe  Standard Class    0.350
Western Europe   Standard Class    0.345
Oceania          Standard Class    0.343
Caribbean        Standard Class    0.322
dtype